# Harmonization: NSL-KDD → ตารางกลาง

Notebook นี้แสดงการแปลงข้อมูล NSL-KDD ที่ผ่าน cleansing แล้วให้ตรงกับ schema กลาง โค้ดสำหรับชุดข้อมูลนี้อยู่ใน notebook โดยตรง ไม่ต้องเปิดไฟล์ฟังก์ชันอื่นเพื่อดูว่าแต่ละคอลัมน์ถูกสร้างอย่างไร

ผลลัพธ์มี 3 ตารางที่เชื่อมด้วย record_id:
- features: ค่าที่อาจใช้เป็น input ของโมเดล
- labels: คำตอบจริงสำหรับประเมินภายหลัง ไม่รวมใน features
- metadata: ที่มาของแถว สถานะความพร้อม และตัวบ่งชี้คุณภาพ

KDDTest-21 เป็นชุดย่อยสำหรับประเมินผล จึงทำเครื่องหมายแถวที่ตรงกันใน KDDTest+ แทนการเพิ่มแถวซ้ำ


## 1. เตรียม path และรายชื่อไฟล์

ค้นหา project root จากตำแหน่ง notebook แทนการเขียน path เฉพาะเครื่อง จากนั้นกำหนดโฟลเดอร์ raw, cleansing และ harmonized ให้เห็นชัดเจน


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data-pipeline" / "sub-dataset" / "NSL-KDD" / "download.py").is_file():
            return candidate
    raise FileNotFoundError("หา project root ไม่พบจากตำแหน่งที่เปิด notebook")

PROJECT_ROOT = find_project_root(Path.cwd())
PIPELINE_DIR = PROJECT_ROOT / "data-pipeline"
DATASET_DIR = PIPELINE_DIR / "sub-dataset" / "NSL-KDD"
RAW_DIR = DATASET_DIR / "raw"
CLEAN_DIR = DATASET_DIR / "output" / "cleansing"
HARMONIZED_DIR = DATASET_DIR / "output" / "harmonized"
HARMONIZED_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Clean input:", CLEAN_DIR)
print("Harmonized output:", HARMONIZED_DIR)


Project: C:\Users\Windows 10\Desktop\CSI_ML_Final
Clean input: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\sub-dataset\NSL-KDD\output\cleansing
Harmonized output: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\sub-dataset\NSL-KDD\output\harmonized


## 2. ตรวจ checksum และจำนวนแถวก่อนแปลง

เทียบ raw กับ manifest ที่สร้างไว้ตอน cleansing และตรวจจำนวนแถวของไฟล์สะอาดกับไฟล์ดิบ การตรวจนี้ทำให้ orig_row_id อ้างกลับไปยังลำดับต้นทางได้อย่างปลอดภัย หากไฟล์ถูกเปลี่ยนหรือจำนวนแถวไม่ตรง notebook จะหยุด ไม่จับคู่ด้วยการเดา


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# ใช้ชนิดไฟล์เดียวกับ manifest ของ pipeline เพื่อให้ Build กลางตรวจสอบต่อได้
raw_paths = sorted(
    path for path in RAW_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".csv", ".gz", ".txt"}
)
if not raw_paths:
    raise FileNotFoundError(f"ไม่พบไฟล์ raw ใน {RAW_DIR}")

current_manifest = pd.DataFrame([
    {
        "file": path.relative_to(RAW_DIR).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in raw_paths
], columns=["file", "bytes", "sha256"])

manifest_path = CLEAN_DIR / "source_manifest.csv"
if not manifest_path.is_file():
    raise FileNotFoundError("ไม่พบ source_manifest.csv กรุณารัน cleansing.ipynb ใหม่ก่อน")

saved_manifest = pd.read_csv(manifest_path).sort_values("file").reset_index(drop=True)
current_manifest = current_manifest.sort_values("file").reset_index(drop=True)
if saved_manifest["file"].tolist() != current_manifest["file"].tolist():
    raise ValueError("รายชื่อ raw files เปลี่ยนหลัง cleansing; กรุณารัน cleansing.ipynb ใหม่")
if saved_manifest["sha256"].tolist() != current_manifest["sha256"].tolist():
    raise ValueError("checksum ของ raw เปลี่ยนหลัง cleansing; กรุณารัน cleansing.ipynb ใหม่")

clean_files = {
    "KDDTrain": "KDDTrain_clean.csv",
    "KDDTest": "KDDTest_clean.csv",
    "KDDTest-21": "KDDTest-21_clean.csv",
}
raw_files = {
    "KDDTrain": "KDDTrain+.txt",
    "KDDTest": "KDDTest+.txt",
    "KDDTest-21": "KDDTest-21.txt",
}
clean_data = {}
for split_name, clean_name in clean_files.items():
    clean_path = CLEAN_DIR / clean_name
    raw_path = RAW_DIR / raw_files[split_name]
    if not clean_path.is_file() or not raw_path.is_file():
        raise FileNotFoundError(f"ไฟล์ของ {split_name} ไม่ครบ: {clean_path} / {raw_path}")

    clean_frame = pd.read_csv(clean_path)
    raw_rows = sum(1 for _ in raw_path.open(encoding="utf-8"))
    if len(clean_frame) != raw_rows:
        raise ValueError(
            f"{split_name}: raw มี {raw_rows:,} แถว แต่ clean มี {len(clean_frame):,} แถว"
        )
    clean_data[split_name] = clean_frame

# ตรวจว่าไฟล์ Train 20% เป็นแถวจาก Train เต็ม ไม่ใช่ข้อมูลอิสระที่ควรนำเข้าซ้ำ
full_train_path = RAW_DIR / "KDDTrain+.txt"
reduced_train_path = RAW_DIR / "KDDTrain+_20Percent.txt"
if full_train_path.is_file() and reduced_train_path.is_file():
    full_train_rows = set(full_train_path.read_text(encoding="utf-8").splitlines())
    reduced_train_rows = reduced_train_path.read_text(encoding="utf-8").splitlines()
    unmatched_rows = [row for row in reduced_train_rows if row not in full_train_rows]
    if unmatched_rows:
        raise ValueError(
            f"พบ {len(unmatched_rows):,} แถวใน KDDTrain+_20Percent ที่ไม่อยู่ใน KDDTrain+"
        )
    print(
        "ตรวจ subset KDDTrain+_20Percent:",
        f"{len(reduced_train_rows):,} แถวอยู่ใน KDDTrain+ ทั้งหมด",
    )

print("ตรวจจำนวนแถว:", {name: len(frame) for name, frame in clean_data.items()})
display(current_manifest)


ตรวจ subset KDDTrain+_20Percent: 25,192 แถวอยู่ใน KDDTrain+ ทั้งหมด
ตรวจจำนวนแถว: {'KDDTrain': 125973, 'KDDTest': 22544, 'KDDTest-21': 11850}


,file,bytes,sha256
0,KDDTest+.txt,3441513,fa46b0935342616aa83b7c2578db355b6a7aaabbc49224...
1,KDDTest-21.txt,1814092,746993ac9e25868827cacf09eab450050a2a1056e1ce48...
2,KDDTrain+.txt,19109424,1b86d2f957b33082081bba410fe129b475efebcc13c901...
3,KDDTrain+_20Percent.txt,3822033,7ea86479faab5ca2190b7f18b4982fb058ce5bf2b46e0e...


## 3.1 ตรวจแถวจาก KDDTest-21

การแปลงใช้ชื่อคอลัมน์จริงของ NSL-KDD เช่น src_bytes และ protocol_type แล้วเปลี่ยนเป็นชื่อกลาง ตัวอย่างเช่น src_bytes → bytes_forward ส่วน label และ attack_family จะแยกเก็บใน labels ไม่ปะปนกับ model features


In [3]:
# ตรวจว่า KDDTest-21 เป็นแถวที่อยู่ใน KDDTest+ จริง โดยไม่สร้างข้อมูลซ้ำ
def row_hashes(frame: pd.DataFrame) -> pd.Series:
    return pd.util.hash_pandas_object(
        frame.astype("string"), index=False
    ).astype("uint64")

test_hashes = row_hashes(clean_data["KDDTest"])
test21_hashes = row_hashes(clean_data["KDDTest-21"])
if test_hashes.duplicated().any() or test21_hashes.duplicated().any():
    raise ValueError("พบแถวซ้ำใน Test หรือ Test-21 ทำให้จับคู่แบบหนึ่งต่อหนึ่งไม่ได้")

test21_hash_set = set(test21_hashes.tolist())
is_test21_row = test_hashes.isin(test21_hash_set)
if int(is_test21_row.sum()) != len(clean_data["KDDTest-21"]):
    raise ValueError("KDDTest-21 จับคู่กับ KDDTest+ ได้ไม่ครบ จึงไม่ทำเครื่องหมาย evaluation view")

print(
    f"KDDTest-21 ที่จับคู่ใน KDDTest+: {int(is_test21_row.sum()):,} "
    f"จาก {len(clean_data['KDDTest-21']):,} แถว"
)

# รวม service และสถานะการเชื่อมต่อที่มีความหมายใกล้กัน
service_aliases = {
    "http": "http", "www": "http",
    "https": "https_ssl", "ssl": "https_ssl", "http_443": "https_ssl",
    "ftp": "ftp", "ftp_data": "ftp", "ftp-data": "ftp",
    "ssh": "ssh", "smtp": "smtp",
    "domain": "dns", "domain_u": "dns", "dns": "dns",
}
connection_states = {
    "sf": "established", "fin": "established", "con": "established",
    "s1": "established", "s2": "established", "s3": "established",
    "s0": "no-reply", "int": "no-reply", "req": "no-reply",
    "rej": "reset", "rsto": "reset", "rstr": "reset",
    "rstos0": "reset", "rst": "reset",
}
family_names = {
    "Normal": "Normal",
    "DoS": "DoS/DDoS",
    "Probe": "Probe/Recon",
    "R2L": "Brute-force/R2L",
    "U2R": "Exploit/U2R",
}


# การโจมตีบางชนิดใน Test ไม่อยู่ใน mapping สั้นของ cleansing
# จัดเข้าตระกูลที่รู้จักจากชื่อ label เดิม โดยยังเก็บ label_raw ไว้ครบ
additional_attack_families = {
    "apache2": "DoS",
    "mailbomb": "DoS",
    "processtable": "DoS",
    "udpstorm": "DoS",
    "worm": "R2L",
    "mscan": "Probe",
    "saint": "Probe",
    "snmpguess": "R2L",
    "snmpgetattack": "R2L",
    "httptunnel": "U2R",
    "named": "R2L",
    "sendmail": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "ps": "U2R",
    "xterm": "U2R",
    "sqlattack": "U2R",
}


KDDTest-21 ที่จับคู่ใน KDDTest+: 11,850 จาก 11,850 แถว


## 3.2 กำหนดวิธีแปลงข้อมูลหนึ่ง split

เตรียมฟังก์ชันอ่านค่าตัวเลขและสร้าง features, labels และ metadata ตามคอลัมน์ที่ NSL-KDD มี


In [4]:
def parse_nonnegative(values: pd.Series) -> tuple[pd.Series, pd.Series]:
    # แปลงเป็นตัวเลข และแยก flag ของค่าที่ติดลบหรือแปลงไม่ได้
    numeric = pd.to_numeric(values, errors="coerce").astype("float64")
    invalid = values.notna() & (
        numeric.isna() | ~np.isfinite(numeric) | numeric.lt(0)
    )
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    numeric.loc[numeric < 0] = np.nan
    return numeric, invalid.astype(bool)


def optional_number(frame: pd.DataFrame, column: str) -> pd.Series:
    # คืนตัวเลขถ้ามีคอลัมน์ มิฉะนั้นเก็บ missing ไว้ ไม่แทนเป็นศูนย์
    if column not in frame:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    numeric = pd.to_numeric(frame[column], errors="coerce").astype("float64")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    numeric.loc[numeric < 0] = np.nan
    return numeric


def build_nsl_tables(frame: pd.DataFrame, split_name: str,
                     test21_view: pd.Series | None = None):
    frame = frame.reset_index(drop=True).copy()

    duration, invalid_duration = parse_nonnegative(frame["duration"])
    bytes_forward, invalid_forward = parse_nonnegative(frame["src_bytes"])
    bytes_backward, invalid_backward = parse_nonnegative(frame["dst_bytes"])

    protocol = frame["protocol_type"].astype("string").str.strip().str.lower()
    protocol = protocol.replace({"6": "tcp", "17": "udp", "1": "icmp"})
    protocol = protocol.where(
        protocol.isna() | protocol.isin(["tcp", "udp", "icmp"]), "other"
    )

    service = frame["service"].astype("string").str.strip().str.lower()
    service_group = service.map(service_aliases).fillna("other")
    service_group = service_group.where(service.notna()).astype("string")

    state = frame["flag"].astype("string").str.strip().str.lower()
    connection_group = state.map(connection_states).fillna("other")
    connection_group = connection_group.where(state.notna()).astype("string")

    record_ids = [
        f"NSL-KDD:{split_name}:{row_number:09d}"
        for row_number in range(len(frame))
    ]
    original_labels = frame["label"].astype("string").str.strip().str.lower()
    source_family = frame["attack_family"].astype("string").str.strip()
    additional_families = original_labels.map(additional_attack_families)
    source_family = source_family.mask(source_family.isna() | source_family.eq("Unknown"), additional_families).fillna("Unknown")
    attack_category = source_family.map(family_names)
    additional_families = original_labels.map(additional_attack_families)
    additional_categories = additional_families.map(family_names)
    attack_category = attack_category.fillna(additional_categories).fillna("Unknown")

    labels = pd.DataFrame({
        "record_id": record_ids,
        "label_raw": original_labels,
        "label_source_category": source_family,
        "attack_category": attack_category.astype("string"),
        "label_binary": pd.to_numeric(frame["is_attack"], errors="raise").astype("int8"),
    })

    features = pd.DataFrame({
        "record_id": record_ids,
        "duration": duration,
        "protocol": protocol,
        "bytes_forward": bytes_forward,
        "bytes_backward": bytes_backward,
        "service_group": service_group,
        "conn_state_group": connection_group,
        "packets_forward": optional_number(frame, "spkts"),
        "packets_backward": optional_number(frame, "dpkts"),
        "mean_packet_size": pd.Series(np.nan, index=frame.index, dtype="float64"),
        "ttl_forward": optional_number(frame, "sttl"),
        "ttl_backward": optional_number(frame, "dttl"),
        "tcp_window_forward": optional_number(frame, "swin"),
        "tcp_window_backward": optional_number(frame, "dwin"),
        "nsl_hot": optional_number(frame, "hot"),
        "nsl_logged_in": optional_number(frame, "logged_in"),
        "nsl_failed_logins": optional_number(frame, "num_failed_logins"),
        "nsl_root_shell": optional_number(frame, "root_shell"),
        "nsl_count_2s": optional_number(frame, "count"),
        "nsl_srv_count_2s": optional_number(frame, "srv_count"),
        "nsl_same_srv_rate_2s": optional_number(frame, "same_srv_rate"),
        "nsl_serror_rate_2s": optional_number(frame, "serror_rate"),
    })

    row_numbers = range(len(frame))
    metadata = pd.DataFrame({
        "record_id": record_ids,
        "source_dataset": "NSL-KDD",
        "orig_row_id": [
            f"{split_name}:{row_number:09d}" for row_number in row_numbers
        ],
        "source_split": "train" if split_name == "KDDTrain" else "test",
        "capture_time": pd.Series(pd.NA, index=frame.index, dtype="string"),
        "split_group": pd.Series(pd.NA, index=frame.index, dtype="string"),
        "dedup_flag": False,
        "sample_weight": 1.0,
        "has_duration": features["duration"].notna(),
        "invalid_duration": invalid_duration,
        "has_protocol": features["protocol"].notna(),
        "has_directional_bytes": features[
            ["bytes_forward", "bytes_backward"]
        ].notna().all(axis=1),
        "invalid_bytes_forward": invalid_forward,
        "invalid_bytes_backward": invalid_backward,
        "has_capture_time": False,
        "has_service": features["service_group"].notna(),
        "has_connection_state": features["conn_state_group"].notna(),
        "has_packet_stats": features[
            ["packets_forward", "packets_backward", "mean_packet_size"]
        ].notna().any(axis=1),
        "has_tcp_details": features[
            ["ttl_forward", "ttl_backward", "tcp_window_forward", "tcp_window_backward"]
        ].notna().any(axis=1),
        "has_nsl_content": features[
            ["nsl_hot", "nsl_logged_in", "nsl_failed_logins", "nsl_root_shell"]
        ].notna().any(axis=1),
        "has_nsl_window": features[
            ["nsl_count_2s", "nsl_srv_count_2s", "nsl_same_srv_rate_2s", "nsl_serror_rate_2s"]
        ].notna().any(axis=1),
        "is_kddtest21_view": (
            test21_view.to_numpy(dtype=bool)
            if test21_view is not None
            else np.zeros(len(frame), dtype=bool)
        ),
    })
    return features, labels, metadata


## 4.1 รวมตารางของแต่ละ split

แปลงข้อมูล train และ test ตามกติกาเฉพาะชุด แล้วรวม features, labels และ metadata เพื่อใช้กฎกลาง


In [ ]:
feature_parts, label_parts, metadata_parts = [], [], []
for split_name in ("KDDTrain", "KDDTest"):
    test21_flags = is_test21_row if split_name == "KDDTest" else None
    feature_table, label_table, metadata_table = build_nsl_tables(
        clean_data[split_name], split_name, test21_flags
    )
    feature_parts.append(feature_table)
    label_parts.append(label_table)
    metadata_parts.append(metadata_table)

features = pd.concat(feature_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)


## 4.2 เตรียมกฎปรับหมวดหมู่และตรวจคุณภาพ

กำหนดชื่อ protocol, service และ connection state ที่ใช้ร่วมกัน พร้อมกฎทำเครื่องหมายแถวที่มีข้อมูลขัดกัน


In [ ]:
CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]


def require(condition, message):
    if not condition:
        raise ValueError(message)


SERVICE_ALIASES = {
    'http':'http', 'www':'http', 'http_443':'https_ssl',
    'https':'https_ssl', 'ssl':'https_ssl',
    'ftp':'ftp', 'ftp_data':'ftp', 'ftp-data':'ftp',
    'ssh':'ssh', 'smtp':'smtp', 'domain':'dns', 'domain_u':'dns', 'dns':'dns',
}


STATE_ALIASES = {
    'sf':'established', 's1':'established', 's2':'established', 's3':'established',
    'fin':'established', 'con':'established', 'established':'established',
    's0':'no-reply', 'req':'no-reply', 'int':'no-reply', 'no-reply':'no-reply',
    'rej':'reset', 'rsto':'reset', 'rstr':'reset', 'rstos0':'reset',
    'rst':'reset', 'reset':'reset',
}


QUALITY_FLAGS = ['invalid_required_value', 'bytes_packet_conflict', 'state_counter_conflict']


def normalize_boolean_metadata(metadata):
    columns=[c for c in metadata if c.startswith(('has_','invalid_')) or c in QUALITY_FLAGS+['quality_ok','service_identified','state_reclassified','dedup_flag','is_kddtest21_view']]
    for col in columns:
        values=metadata[col].astype('string').str.lower()
        if not values.isin(['true','false','1','0']).all():raise ValueError(f'Invalid boolean metadata: {col}')
        metadata[col]=values.map({'true':True,'false':False,'1':True,'0':False}).astype(bool)
    return metadata


def text(values):
    return values.astype('string').str.strip().str.lower().replace({'':'', '-':'-'})


def normalize_protocol(values):
    values=text(values).replace({'6':'tcp','6.0':'tcp','17':'udp','17.0':'udp',
                                 '1':'icmp','1.0':'icmp','58':'icmp','58.0':'icmp','ipv6-icmp':'icmp'})
    present=values.notna() & ~values.isin(['','-'])
    return values.where(values.isin(['tcp','udp','icmp']),'other').where(present,pd.NA)


def port_to_service(value):
    if pd.isna(value) or str(value).strip()=='':return 'other'
    token=str(value).strip()
    try:
        number=int(token,16) if token.lower().startswith('0x') else float(token)
    except (ValueError,TypeError) as error:
        raise ValueError(f'Invalid destination port: {value!r}') from error
    if not np.isfinite(number) or number!=int(number) or not 0<=number<=65535:
        raise ValueError(f'Invalid destination port: {value!r}')
    port=int(number)
    return {53:'dns',20:'ftp',21:'ftp',80:'http',8080:'http',443:'https_ssl',
            25:'smtp',465:'smtp',587:'smtp',22:'ssh'}.get(port,'other')


def quality_flags(features, metadata):
    f=features
    invalid=pd.Series(False,index=f.index)
    for col in ['duration','bytes_forward','bytes_backward']:
        values=pd.to_numeric(f[col],errors='coerce')
        invalid |= values.isna() | ~np.isfinite(values) | values.lt(0)
    invalid |= f.protocol.isna() | f.protocol.astype('string').str.strip().isin(['','-'])
    packet_conflict=pd.Series(False,index=f.index)
    for direction in ['forward','backward']:
        col='packets_'+direction
        if col in f:
            values=pd.to_numeric(f[col],errors='coerce')
            packet_conflict |= values.eq(0) & f['bytes_'+direction].gt(0)
            invalid |= f[col].notna() & (~np.isfinite(values) | values.lt(0))
    raw_state=text(metadata.get('source_connection_state',pd.Series(pd.NA,index=f.index,dtype='string')))
    has_response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:has_response |= f.packets_backward.gt(0)
    # REQ/INT can have response traffic. Literal no-reply/S0 with a response is
    # contradictory source evidence; do not silently erase that evidence.
    state_conflict=raw_state.isin(['no-reply','s0']) & has_response
    return pd.DataFrame({'invalid_required_value':invalid,
                         'bytes_packet_conflict':packet_conflict,
                         'state_counter_conflict':state_conflict},index=f.index)


## 4.3 กำหนดการแปลง label เป็นคลาสกลาง

จับคู่ label ต้นทางกับรหัสคลาสและตรวจว่าตรงกับ label แบบปกติหรือโจมตี


In [ ]:
TACTICS = ['Baseline Operations','TA0040: Impact','TA0043/TA0007: Recon & Discovery','TA0001/TA0004: Initial Access & PrivEsc','TA0006: Credential Access','TA0011/TA0008: C2 & Lateral Movement']


COMMON_MAPPING = {'normal':0,'benign':0,'dos':1,'dos/ddos':1,'ddos':1,'probe':2,'probe/recon':2,'reconnaissance':2,'analysis':2,'exploits':3,'exploit/u2r':3,'shellcode':3,'u2r':3,'r2l':4,'brute-force/r2l':4,'generic':5,'generic/fuzzer':5,'fuzzers':5,'backdoor':5,'backdoors':5,'botnet/backdoor':5,'worms':5}


SOURCE_MAPPING = {
    'NSL-KDD': {k:COMMON_MAPPING[k] for k in ['normal','dos','probe','r2l','u2r']},
    'UNSW-NB15': {k:COMMON_MAPPING[k] for k in ['normal','analysis','backdoor','backdoors','dos','exploits','fuzzers','generic','reconnaissance','shellcode','worms']},
    'CIC-IDS2017': {**COMMON_MAPPING, 'dos hulk':1,'dos goldeneye':1,'dos slowloris':1,'dos slowhttptest':1,'heartbleed':3,'portscan':2,'ftp-patator':4,'ssh-patator':4,'web attack - brute force':4,'web attack - xss':3,'web attack - sql injection':3,'web attack brute force':4,'web attack xss':3,'web attack sql injection':3,'infiltration':3,'bot':5},
    # Project-level category grouping, not per-record MITRE attribution.
    'BoT-IoT': {**COMMON_MAPPING,'theft':5,'data_theft':5,'bruteforce':4},
}


def map_labels(labels, metadata):
    result=labels.copy(); ids=pd.Series(index=labels.index,dtype='int64')
    for source,index in metadata.groupby('source_dataset').groups.items():
        require(source in SOURCE_MAPPING,f'Unsupported source {source}')
        values=labels.loc[index,'label_source_category'].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
        mapped=values.map(SOURCE_MAPPING[source])
        require(mapped.notna().all(),f'Unmapped labels in {source}: {values[mapped.isna()].value_counts().to_dict()}')
        ids.loc[index]=mapped.astype(int)
    ids=ids.astype(int)
    require(labels.label_binary.eq(ids.ne(0).astype(int)).all(),'Mapped categories contradict original binary truth')
    result['attack_category_id']=ids;result['attack_category']=ids.map(dict(enumerate(CLASS_NAMES)));result['mitre_tactic']=ids.map(dict(enumerate(TACTICS)))
    return result


## 4.4 เตรียมฟังก์ชัน harmonization กลาง

กำหนดขั้นนำกฎกลางไปใช้กับตารางทั้งสาม รวมถึงข้อมูลกำกับว่าแต่ละค่ามาจากอะไร


In [ ]:
def harmonize_tables(dataset, features, labels, metadata, source):
    f=features.reset_index(drop=True).copy()
    l=labels.reset_index(drop=True).copy()
    m=metadata.reset_index(drop=True).copy()
    source=source.reset_index(drop=True)
    if not len(source)==len(f)==len(l)==len(m):raise ValueError('Source/output row count mismatch')
    for col in ['packets_forward','packets_backward']:
        if col not in f:f[col]=np.nan
    protocol_col={'NSL-KDD':'protocol_type','UNSW-NB15':'proto','CIC-IDS2017':'protocol','BoT-IoT':'proto'}[dataset]
    state_col={'NSL-KDD':'flag','UNSW-NB15':'state','CIC-IDS2017':'flow_state','BoT-IoT':'state'}[dataset]
    m['source_protocol']=source[protocol_col].astype('string')
    m['source_connection_state']=source[state_col].astype('string')
    f['protocol']=normalize_protocol(source[protocol_col])
    if dataset in ['NSL-KDD','UNSW-NB15']:
        service=text(source['service'])
        present=service.notna() & ~service.isin(['','-'])
        f['service_group']=service.map(SERVICE_ALIASES).fillna('other')
        m['source_service']=source['service'].astype('string')
        m['service_basis']='source_service_name'
        m['service_identified']=present
        m['has_service']=present
    else:
        port='dest_port' if dataset=='CIC-IDS2017' else 'dport'
        values=source[port] if port in source else pd.Series(pd.NA,index=source.index,dtype='string')
        m['source_service']=values.astype('string')
        if dataset=='BoT-IoT':
            # ARP has no transport port. The mirror represents it as -1.
            missing_arp_port=text(source[protocol_col]).eq('arp') & text(values).isin(['-1','-1.0'])
            values=values.mask(missing_arp_port.fillna(False))
        present=values.notna() & values.astype('string').str.strip().ne('')
        f['service_group']=values.map(port_to_service)
        m['service_basis']=np.where(present,'destination_port_hint','unavailable')
        m['service_identified']=False  # Port 443 is a hint, not proof of TLS.
        m['has_service']=present
    state=text(source[state_col])
    present_state=state.notna() & ~state.isin(['','-'])
    base=state.map(STATE_ALIASES).fillna('other')
    response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:response |= f.packets_backward.gt(0)
    f['conn_state_group']=base.mask(base.eq('no-reply') & response,'other')
    m['state_reclassified']=base.ne(f.conn_state_group)
    m['connection_state_basis']='source_state_and_directional_counters'
    m['has_connection_state']=present_state
    m['has_duration']=f.duration.notna()
    m['has_protocol']=f.protocol.notna()
    m['has_directional_bytes']=f[['bytes_forward','bytes_backward']].notna().all(axis=1)
    m['has_packet_stats']=f[['packets_forward','packets_backward']].notna().all(axis=1)
    for col,out in [('duration','invalid_duration'),('bytes_forward','invalid_bytes_forward'),('bytes_backward','invalid_bytes_backward')]:
        values=pd.to_numeric(f[col],errors='coerce')
        m[out]=values.isna() | ~np.isfinite(values) | values.lt(0)
    flags=quality_flags(f,m)
    for col in QUALITY_FLAGS:m[col]=flags[col]
    m['quality_ok']=~flags.any(axis=1)
    m['quality_issues']=[';'.join(c for c in QUALITY_FLAGS if row[c]) for row in flags.to_dict('records')]
    m['duration_unit']='seconds'
    m=normalize_boolean_metadata(m)
    l['source_attack_category']=l.attack_category.astype('string')
    l=map_labels(l,m)  # One taxonomy for source tables and CENTRAL.
    return f,l,m


_port_to_service = port_to_service


## 4.5 ใช้กฎกลางและตรวจตาราง

เรียก harmonization กับตารางที่รวมแล้ว ตรวจ record_id และดูจำนวนแถวของแต่ละคลาส


In [ ]:
source_frame = pd.concat([clean_data["KDDTrain"], clean_data["KDDTest"]], ignore_index=True)
features, labels, metadata = harmonize_tables("NSL-KDD", features, labels, metadata, source_frame)

# ตรวจการเชื่อมสามตารางก่อนเขียนไฟล์
for table_name, table in {
    "features": features, "labels": labels, "metadata": metadata
}.items():
    if table["record_id"].isna().any() or table["record_id"].duplicated().any():
        raise ValueError(f"{table_name}: record_id หายหรือซ้ำ")

if not features["record_id"].equals(labels["record_id"]):
    raise ValueError("record_id ใน features และ labels เรียงไม่ตรงกัน")
if not features["record_id"].equals(metadata["record_id"]):
    raise ValueError("record_id ใน features และ metadata เรียงไม่ตรงกัน")

forbidden = {
    "label", "label_raw", "attack_category", "label_binary", "is_attack",
    "source_dataset", "orig_row_id", "capture_time", "split_group",
    "sample_weight", "dedup_flag", "difficulty",
}
leaked_columns = forbidden.intersection(features.columns)
if leaked_columns:
    raise ValueError(f"พบ label/metadata ใน features: {sorted(leaked_columns)}")

print("จำนวนแถวในแต่ละตาราง:", {
    "features": len(features), "labels": len(labels), "metadata": len(metadata)
})
print("จำนวน model feature:", len(features.columns) - 1, "(ไม่รวม record_id)")
display(labels["attack_category"].value_counts().rename_axis("attack_category").to_frame("rows"))


## 5. บันทึกผลและรายงานที่มา\n\nเขียน CSV แยกสามตาราง พร้อมรายงานจำนวนแถว ความพร้อมของ feature และ provenance ซึ่งบอกเวอร์ชัน contract กับ checksum ของ raw ที่ใช้

In [6]:
tables = {
    "features": features,
    "labels": labels,
    "metadata": metadata,
}
for table_name, table in tables.items():
    temporary_path = HARMONIZED_DIR / f".{table_name}.partial.csv"
    final_path = HARMONIZED_DIR / f"{table_name}.csv"
    table.to_csv(temporary_path, index=False, encoding="utf-8")
    temporary_path.replace(final_path)

harmonization_report = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in tables.items()
])
harmonization_report.to_csv(
    HARMONIZED_DIR / "harmonization_report.csv", index=False
)

availability_rows = []
for column in features.columns:
    if column == "record_id":
        continue
    available = int(features[column].notna().sum())
    availability_rows.append({
        "source_dataset": "NSL-KDD",
        "feature": column,
        "rows_available": available,
        "rows_total": len(features),
        "availability_percent": 100 * available / len(features) if len(features) else 0,
    })
pd.DataFrame(availability_rows).to_csv(
    HARMONIZED_DIR / "feature_availability.csv", index=False
)

quality_report = pd.DataFrame([{
    "source_dataset": "NSL-KDD",
    "rows": len(metadata),
    "duplicate_rows": int(metadata["dedup_flag"].sum()),
    "invalid_duration": int(metadata["invalid_duration"].sum()),
    "invalid_bytes_forward": int(metadata["invalid_bytes_forward"].sum()),
    "invalid_bytes_backward": int(metadata["invalid_bytes_backward"].sum()),
}])
quality_report.to_csv(HARMONIZED_DIR / "harmonization_quality.csv", index=False)

current_manifest.to_csv(HARMONIZED_DIR / "source_manifest.csv", index=False)
registry = json.loads(
    (PIPELINE_DIR / "CENTRAL" / "dataset_registry.json").read_text(encoding="utf-8")
)
contract = json.loads(
    (PIPELINE_DIR / "CENTRAL" / "feature_contract.json").read_text(encoding="utf-8")
)
provenance = {
    "dataset": "NSL-KDD",
    "source_version": registry["datasets"]["NSL-KDD"]["version"],
    "registry_version": registry["registry_version"],
    "feature_contract_version": contract["contract_version"],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "row_count": len(features),
    "source_files": current_manifest.to_dict(orient="records"),
}
(HARMONIZED_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(harmonization_report)
print("ผล Harmonization:", HARMONIZED_DIR)


,table,rows,columns
0,features,148517,22
1,labels,148517,5
2,metadata,148517,22


ผล Harmonization: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\sub-dataset\NSL-KDD\output\harmonized


## 6. ดู mapping, feature ที่ไม่มีใน NSL และ taxonomy label

ตารางแสดงคอลัมน์ต้นทาง → คอลัมน์กลาง รวมทั้งระบุคอลัมน์ที่ NSL-KDD ไม่มีจริง เช่น packet count, TTL และ TCP window จึงเก็บเป็น missing พร้อม availability flag ไม่สร้างศูนย์ปลอม อีกตารางแสดง label ดั้งเดิมและกลุ่มโจมตีที่ map ได้ เพื่อให้ตรวจ `Unknown` และการคง label เดิมได้

In [7]:
nsl_mapping_specs = [
    ('duration', 'features.duration', 'ตัวเลขวินาที; ค่าติดลบ/ผิดรูปแบบ → missing + invalid flag'),
    ('protocol_type', 'features.protocol', 'ปรับตัวพิมพ์และ map เป็น tcp/udp/icmp/other'),
    ('src_bytes', 'features.bytes_forward', 'bytes จาก source → ทิศทางต้นทาง'),
    ('dst_bytes', 'features.bytes_backward', 'bytes จาก destination → ทิศทางกลับ'),
    ('service', 'features.service_group', 'รวม alias เป็นกลุ่มบริการกลาง; ค่าไม่อยู่ในกลุ่ม→other'),
    ('flag', 'features.conn_state_group', 'รวม connection flag เป็น established/no-reply/reset/other'),
    ('ไม่มีใน NSL-KDD', 'features.packets_forward / packets_backward / mean_packet_size', 'ไม่สร้างขึ้นเอง; เก็บ missing'),
    ('ไม่มีใน NSL-KDD', 'features.ttl_forward / ttl_backward / tcp_window_forward / tcp_window_backward', 'ไม่สร้างขึ้นเอง; เก็บ missing'),
    ('hot / logged_in / num_failed_logins / root_shell', 'features.nsl_hot / nsl_logged_in / nsl_failed_logins / nsl_root_shell', 'คง feature เนื้อหาเฉพาะ NSL'),
    ('count / srv_count / same_srv_rate / serror_rate', 'features.nsl_count_2s / nsl_srv_count_2s / nsl_same_srv_rate_2s / nsl_serror_rate_2s', 'คงตัวชี้วัดหน้าต่าง 2 วินาทีตาม schema NSL'),
    ('label', 'labels.label_raw', 'เก็บ label ชนิดการโจมตีเดิมไว้'),
    ('attack_family', 'labels.label_source_category / attack_category', 'map DoS/Probe/R2L/U2R เป็น taxonomy กลาง; label เดิมยังอยู่'),
    ('is_attack', 'labels.label_binary', 'เก็บ target 0/1 แยกจาก model features'),
    ('ลำดับแถวเดิม + split', 'metadata.orig_row_id / source_split', 'สร้าง record_id ที่ย้อนกลับไปยัง split/แถวได้'),
]
available_rows = []
for _, row in pd.DataFrame(nsl_mapping_specs, columns=['source', 'target', 'rule']).iterrows():
    target_details = []
    current_table = None
    for target_name in row['target'].replace('+', '/').split('/'):
        target_name = target_name.strip()
        if '.' in target_name:
            current_table, target_column = target_name.split('.', 1)
        else:
            target_column = target_name
        target_frame = {'features': features, 'labels': labels, 'metadata': metadata}.get(current_table)
        if target_frame is not None and target_column in target_frame.columns:
            available = int(target_frame[target_column].notna().sum())
            target_details.append(f'{current_table}.{target_column}: {available:,}/{len(features):,} แถว')
        else:
            target_details.append(f'{target_name}: ไม่มีคอลัมน์นี้')
    available_rows.append({'source_column': row['source'], 'central_column': row['target'],
                           'transformation': row['rule'],
                           'output_coverage': '; '.join(target_details)})
nsl_mapping_detail = pd.DataFrame(available_rows)
display(nsl_mapping_detail)
print('label ดั้งเดิม → กลุ่มที่ใช้ประเมิน/อธิบาย:')
display(labels.groupby(['label_raw', 'label_source_category', 'attack_category', 'label_binary'], dropna=False)
        .size().rename('rows').reset_index().sort_values('rows', ascending=False).head(50))
print('การครอบคลุมค่ากลุ่ม feature:')
display(pd.DataFrame([
    {'feature_group': 'เวลาจับข้อมูล', 'available_rows': int(metadata['has_capture_time'].sum()), 'total_rows': len(metadata)},
    {'feature_group': 'สถิติ packet', 'available_rows': int(metadata['has_packet_stats'].sum()), 'total_rows': len(metadata)},
    {'feature_group': 'รายละเอียด TCP', 'available_rows': int(metadata['has_tcp_details'].sum()), 'total_rows': len(metadata)},
    {'feature_group': 'เนื้อหาเฉพาะ NSL', 'available_rows': int(metadata['has_nsl_content'].sum()), 'total_rows': len(metadata)},
    {'feature_group': 'สถิติ window เฉพาะ NSL', 'available_rows': int(metadata['has_nsl_window'].sum()), 'total_rows': len(metadata)},
    {'feature_group': 'มุมมอง KDDTest-21', 'available_rows': int(metadata['is_kddtest21_view'].sum()), 'total_rows': len(metadata)},
]))
print('คุณภาพตัวเลขหลังแปลง (invalid จะคงเป็น missing พร้อม flag; ไม่มีแถวถูกลบ):')
display(quality_report)

,source_column,central_column,transformation,output_coverage
0,duration,features.duration,ตัวเลขวินาที; ค่าติดลบ/ผิดรูปแบบ → missing + i...,"features.duration: 148,517/148,517 แถว"
1,protocol_type,features.protocol,ปรับตัวพิมพ์และ map เป็น tcp/udp/icmp/other,"features.protocol: 148,517/148,517 แถว"
2,src_bytes,features.bytes_forward,bytes จาก source → ทิศทางต้นทาง,"features.bytes_forward: 148,517/148,517 แถว"
3,dst_bytes,features.bytes_backward,bytes จาก destination → ทิศทางกลับ,"features.bytes_backward: 148,517/148,517 แถว"
4,service,features.service_group,รวม alias เป็นกลุ่มบริการกลาง; ค่าไม่อยู่ในกลุ...,"features.service_group: 148,517/148,517 แถว"
5,flag,features.conn_state_group,รวม connection flag เป็น established/no-reply/...,"features.conn_state_group: 148,517/148,517 แถว"
6,ไม่มีใน NSL-KDD,features.packets_forward / packets_backward / ...,ไม่สร้างขึ้นเอง; เก็บ missing,"features.packets_forward: 0/148,517 แถว; featu..."
7,ไม่มีใน NSL-KDD,features.ttl_forward / ttl_backward / tcp_wind...,ไม่สร้างขึ้นเอง; เก็บ missing,"features.ttl_forward: 0/148,517 แถว; features...."
8,hot / logged_in / num_failed_logins / root_shell,features.nsl_hot / nsl_logged_in / nsl_failed_...,คง feature เนื้อหาเฉพาะ NSL,"features.nsl_hot: 148,517/148,517 แถว; feature..."
9,count / srv_count / same_srv_rate / serror_rate,features.nsl_count_2s / nsl_srv_count_2s / nsl...,คงตัวชี้วัดหน้าต่าง 2 วินาทีตาม schema NSL,"features.nsl_count_2s: 148,517/148,517 แถว; fe..."


label ดั้งเดิม → กลุ่มที่ใช้ประเมิน/อธิบาย:


,label_raw,label_source_category,attack_category,label_binary,rows
16,normal,Normal,Normal,0,77054
14,neptune,DoS,DoS/DDoS,1,45871
25,satan,Probe,Probe/Recon,1,4368
7,ipsweep,Probe,Probe/Recon,1,3740
27,smurf,DoS,DoS/DDoS,1,3311
20,portsweep,Probe,Probe/Recon,1,3088
15,nmap,Probe,Probe/Recon,1,1566
1,back,DoS,DoS/DDoS,1,1315
4,guess_passwd,R2L,Brute-force/R2L,1,1284
11,mscan,Probe,Probe/Recon,1,996


การครอบคลุมค่ากลุ่ม feature:


,feature_group,available_rows,total_rows
0,เวลาจับข้อมูล,0,148517
1,สถิติ packet,0,148517
2,รายละเอียด TCP,0,148517
3,เนื้อหาเฉพาะ NSL,148517,148517
4,สถิติ window เฉพาะ NSL,148517,148517
5,มุมมอง KDDTest-21,11850,148517


คุณภาพตัวเลขหลังแปลง (invalid จะคงเป็น missing พร้อม flag; ไม่มีแถวถูกลบ):


,source_dataset,rows,duplicate_rows,invalid_duration,invalid_bytes_forward,invalid_bytes_backward
0,NSL-KDD,148517,0,0,0,0


## 7. เตรียมรายงานคุณภาพ

กำหนดโครงสร้างรายงานคุณภาพและวิธีเขียนตารางแยกแถวที่ถูกกักไว้


In [ ]:
PROJECT_ROOT = PIPELINE_DIR.parent

NUMERIC = ["duration", "bytes_forward", "bytes_backward"]


CATEGORIES = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}


CORE = NUMERIC + list(CATEGORIES)


def write_quality_outputs(dataset_dir, f,l,m):
    from pathlib import Path
    d=Path(dataset_dir)/'output/harmonized'
    bad=~m.quality_ok
    q=d/'quarantine';q.mkdir(parents=True,exist_ok=True)
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        frame.loc[bad].to_csv(q/(name+'.csv'),index=False)
    rows=[]
    for split,block in m.groupby('source_split'):
        row={'source_dataset':dataset_dir.name,'split':split,'rows':len(block),
             'eligible_rows':int(block.quality_ok.sum()),'quarantined_rows':int((~block.quality_ok).sum())}
        for col in QUALITY_FLAGS+['invalid_duration','invalid_bytes_forward','invalid_bytes_backward','state_reclassified']:
            row[col]=int(block[col].sum())
        rows.append(row)
    pd.DataFrame(rows).to_csv(d/'harmonization_quality.csv',index=False)
    path=d/'provenance.json'
    import json
    provenance=json.loads(path.read_text(encoding='utf8')) if path.exists() else {}
    provenance.update({'harmonization_schema_version':6,'tables_verified':True,
                       'eligible_rows':int((~bad).sum()),'quarantined_rows':int(bad.sum()),
                       'quality_policy':'Retain all source records; exclude flagged records from CENTRAL and keep separate quarantine tables'})
    write_json(path,provenance)


def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x)),encoding='utf8')


def raw_manifest(dataset_dir):
    return [{'file':p.relative_to(Path(dataset_dir)/'raw').as_posix(),'bytes':p.stat().st_size,'sha256':digest(p)} for p in sorted((Path(dataset_dir)/'raw').rglob('*')) if p.is_file() and p.suffix.lower() in {'.csv','.gz','.txt'}]


def verify_manifest(dataset_dir, manifest):
    saved=pd.read_csv(manifest).sort_values('file').reset_index(drop=True)
    now=pd.DataFrame(raw_manifest(dataset_dir)).sort_values('file').reset_index(drop=True)
    require(len(now)>0 and saved[['file','sha256']].equals(now[['file','sha256']]),f'Stale raw manifest: {manifest}; rerun cleansing/harmonization')


## 8. เตรียมการตรวจผลลัพธ์และ dependency stamp

กำหนดวิธีอ่านตารางกลับมาตรวจและเก็บ checksum ของไฟล์ที่ใช้


In [ ]:
def read_tables(folder):
    # Source labels can be numeric (UNSW) or text (NSL/CIC). Preserve lexical
    # identifiers consistently across source tables, concatenation and CSV reload.
    types={name:'string' for name in ['record_id','label_raw','label_source_category','source_attack_category','orig_row_id','split_group','source_dataset','source_split','source_protocol','source_connection_state','source_service','service_basis','connection_state_basis','duration_unit','quality_issues','capture_time']}
    f,l,m=tuple(pd.read_csv(Path(folder)/(t+'.csv'),low_memory=False,dtype=types) for t in ['features','labels','metadata'])
    return f,l,normalize_boolean_metadata(m)


def validate_tables(f,l,m,allow_quarantined=False):
    require(len(f)>0 and len(f)==len(l)==len(m),'Empty tables or row-count mismatch')
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        require('record_id' in frame and frame.record_id.notna().all() and frame.record_id.astype(str).str.strip().ne('').all() and frame.record_id.is_unique,f'{name}: missing/duplicate record_id')
    require(f.record_id.equals(l.record_id) and f.record_id.equals(m.record_id),'Record IDs/order mismatch')
    require(not set(f.columns).intersection({'label','label_raw','label_source_category','attack_category','attack_category_id','label_binary','mitre_tactic','is_attack','source_dataset','sample_weight','data_split','is_synthetic'}),'Target/metadata leakage in features')
    require(set(CORE).issubset(f.columns),'Missing core columns')
    computed=quality_flags(f,m)
    eligible=~computed.any(axis=1)
    if 'quality_ok' in m:
        require(m.quality_ok.notna().all() and m.quality_ok.eq(eligible).all(),'quality_ok disagrees with recomputed quality rules')
        for col in computed:require(col in m and m[col].eq(computed[col]).all(),f'{col}: quality flag mismatch')
    require(allow_quarantined or eligible.all(),'Invalid duration/bytes/protocol or cross-field conflicts: quarantined records are not allowed in model splits')
    checked=f.loc[eligible] if allow_quarantined else f
    for c in NUMERIC:
        z=pd.to_numeric(checked[c],errors='coerce')
        require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: missing, infinite or negative input')
    for c,values in CATEGORIES.items():require(checked[c].notna().all() and checked[c].isin(values).all(),f'{c}: missing/unknown categories')
    response=checked.bytes_backward.gt(0)
    if 'packets_backward' in checked:response |= checked.packets_backward.gt(0)
    require(not (checked.conn_state_group.eq('no-reply') & response).any(),'no-reply contradicts backward traffic')
    for c in ['packets_forward','packets_backward','mean_packet_size','ttl_forward','ttl_backward','tcp_window_forward','tcp_window_backward']:
        if c in checked:
            z=checked[c].dropna()
            require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: invalid measured extension')
    require('label_binary' in l and l.label_binary.isin([0,1]).all(),'Invalid binary labels')
    if 'attack_category_id' in l:
        require(l.attack_category_id.isin(range(6)).all(),'Invalid central classes')
        require(l.label_binary.eq(l.attack_category_id.ne(0).astype(int)).all(),'Binary/multiclass labels disagree')
        require(l.attack_category.eq(l.attack_category_id.map(dict(enumerate(CLASS_NAMES)))).all(),'Class name/ID mismatch')
    if 'sample_weight' in m:require(np.isfinite(m.sample_weight).all() and m.sample_weight.gt(0).all(),'Invalid sample weights')
    for flag in ['invalid_duration','invalid_bytes_forward','invalid_bytes_backward']:
        if flag in m:require(not m.loc[eligible,flag].astype(str).str.lower().isin(['true','1']).any(),f'{flag}: flagged invalid rows')


def code_signature(pipeline_dir):
    p=Path(pipeline_dir)
    files=sorted([*p.glob('*.py'),*p.glob('CENTRAL/*.json'),*p.glob('CENTRAL/*.ipynb'),*p.glob('sub-dataset/*/*.ipynb'),*p.glob('sub-dataset/*/*.py')])
    signature = {f.relative_to(p).as_posix():digest(f) for f in files}
    shared = PROJECT_ROOT / "webapp/Dashboard/backend/src/models/central_contract.py"
    signature["../webapp/Dashboard/backend/src/models/central_contract.py"] = digest(shared)
    return signature


def stamp_dataset(dataset_dir):
    d=Path(dataset_dir);p=d.parent.parent
    f,l,m=read_tables(d/'output/harmonized');validate_tables(f,l,m,allow_quarantined=True)
    verify_manifest(d,d/'output/cleansing/source_manifest.csv')
    verify_manifest(d,d/'output/harmonized/source_manifest.csv')
    cleaned=cleaned_fingerprints(d)
    if 'quality_ok' in m:write_quality_outputs(d,f,l,m)
    origin=d/'raw/source_provenance.json'
    write_json(d/'output/harmonized/dependencies.json',{'raw':raw_manifest(d),'code':code_signature(p),'cleaned':cleaned,'source_provenance':digest(origin) if origin.exists() else None,'tables':{t:digest(d/'output/harmonized'/f'{t}.csv') for t in ['features','labels','metadata']}})


def cleaned_fingerprints(dataset_dir):
    d=Path(dataset_dir)
    names={'NSL-KDD':['KDDTrain_clean.csv','KDDTest_clean.csv','KDDTest-21_clean.csv'],
           'UNSW-NB15':['UNSW_NB15_training-set.csv','UNSW_NB15_testing-set.csv'],
           'CIC-IDS2017':['CIC_IDS2017_training-set.csv','CIC_IDS2017_testing-set.csv'],
           'BoT-IoT':['BoT_IoT_training-set.csv','BoT_IoT_testing-set.csv']}[d.name]
    for name in names:require((d/'output/cleansing'/name).is_file(),f'Missing cleaned file: {d.name}/{name}')
    return {name:digest(d/'output/cleansing'/name) for name in names}


## 9. ตรวจไฟล์และบันทึกผลยืนยัน

ตรวจไฟล์ที่เพิ่งเขียน สร้างรายงานคุณภาพ และบันทึก dependencies.json ให้ CENTRAL ใช้ตรวจที่มา


In [ ]:
stamp_dataset(DATASET_DIR)
